# Accessible Sidewalk Assessment: train on Colab

Runs top to bottom: get the code, download and crop the data, train, evaluate.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU`.
2. Upload `sidewalk-code.zip` (made by `python colab/package_code.py`) to a folder called `sidewalk` in your Google Drive, so it sits at `My Drive/sidewalk/sidewalk-code.zip`.
3. `Runtime > Run all`. You'll be asked once to allow access to Google Drive.

**Everything that should survive a disconnect goes to `My Drive/sidewalk/`:** the cropped images (about 4 GB, one `.tar` per category), checkpoints and results. If the session drops, reconnect and `Run all` again: finished categories come back from Drive in a few minutes, and training continues from the last finished epoch.

## 1. Settings

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/sidewalk'  # sidewalk-code.zip goes here; crops, checkpoints and results are saved here

# Which model to train. '' is the repo default: the CNN from classifier.py, 60 epochs from scratch.
# Other options, all from the README:
#   '--backbone resnet50'                      ResNet-50, ImageNet weights, 15 epochs (fastest good first result)
#   '--backbone resnet50 --loss weighted_bce'  or --loss focal, or --oversample
#   '--backbone dinov2 --linear-probe'         or drop --linear-probe for full fine-tuning; same for clip
TRAIN_ARGS = ''

## 2. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.'
print(torch.cuda.get_device_name(0))

## 3. Get the code from Drive

In [ ]:
import os, shutil, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
CODE_ZIP = f'{DRIVE_DIR}/sidewalk-code.zip'
assert os.path.exists(CODE_ZIP), f'Upload sidewalk-code.zip to {DRIVE_DIR} in Google Drive first.'

shutil.rmtree('/content/code', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
os.chdir('/content/code')
sys.path.insert(0, '/content/code')

!pip install -q -r requirements.txt
!pip install -q huggingface_hub hf_xet

result = subprocess.run([sys.executable, 'tests/smoke_test.py'], capture_output=True, text=True)
assert result.returncode == 0, result.stdout[-2000:] + result.stderr[-3000:]
print('Code ready, smoke test passed.')

## 4. Download and crop the data

For each category: if its crops are already in Drive, unpack them. Otherwise download the zip from Hugging Face, crop the images straight out of it to 640x640 JPEGs, save the crops to Drive, and delete the zip. The zips total 71 GB, but only one is on disk at a time.

First time: roughly 30 to 60 minutes. Later sessions: a few minutes.

In [ ]:
import os, shutil, subprocess, sys, tarfile, time
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'  # don't keep a second copy of the zips in a download cache
from huggingface_hub import hf_hub_download

DATA_DIR = '/content/data/processed'
ZIP_DIR = '/content/zips'
CACHE_DIR = f'{DRIVE_DIR}/processed'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def free_gb():
    return shutil.disk_usage('/content').free / 1e9

for category in ['Crosswalk', 'Obstacle', 'SurfaceProblem', 'CurbRamp']:  # smallest first
    start = time.time()
    tar_path = f'{CACHE_DIR}/{category}.tar'
    if os.path.exists(f'{DATA_DIR}/{category}/test.csv'):  # prepare_data.py writes this last
        print(f'{category}: already on disk')
        continue
    if os.path.exists(tar_path):
        with tarfile.open(tar_path) as archive:
            archive.extractall(DATA_DIR)
        print(f'{category}: restored from Drive in {time.time() - start:.0f}s')
        continue

    print(f'{category}: downloading ({free_gb():.0f} GB free on disk)')
    zip_path = hf_hub_download('projectsidewalk/sidewalk-tagger-ai-validated', f'Validated/{category}.zip',
                               repo_type='dataset', local_dir=ZIP_DIR)
    print(f'{category}: downloaded in {time.time() - start:.0f}s, cropping')
    subprocess.run([sys.executable, 'prepare_data.py', '--zip-dir', os.path.dirname(zip_path),
                    '--output-dir', DATA_DIR, '--categories', category, '--jpeg', '--workers', '8'], check=True)
    os.remove(zip_path)

    with tarfile.open(tar_path + '.partial', 'w') as archive:  # renamed when complete, so a half-written tar is never used
        archive.add(f'{DATA_DIR}/{category}', arcname=category)
    os.replace(tar_path + '.partial', tar_path)
    print(f'{category}: done in {(time.time() - start) / 60:.1f} min, saved to Drive ({os.path.getsize(tar_path) / 1e9:.1f} GB)')

shutil.rmtree(ZIP_DIR, ignore_errors=True)
print(f'All data ready. {free_gb():.0f} GB free on disk.')

## 5. Train

Prints one line per epoch with validation macro F1 and how long the epoch took. Multiply that by the number of epochs to see how long the run will take. The best epoch is saved to `My Drive/sidewalk/checkpoints/`.

If the session disconnects, reconnect and `Run all` again: it carries on from the last finished epoch. Running this cell again after training has finished does nothing.

In [ ]:
import os, shlex
import train

WORKERS = os.cpu_count()
train_argv = ['--data-dir', DATA_DIR, '--output-dir', f'{DRIVE_DIR}/checkpoints', '--num-workers', str(WORKERS),
              '--resume'] + shlex.split(TRAIN_ARGS)
args = train.parse_args(train_argv)
CHECKPOINT = os.path.join(args.output_dir, f'{train.run_name(args)}.pth')
print(f'Training {train.run_name(args)}: {args.epochs} epochs, batch size {args.batch_size}, image size {args.image_size}')

!python train.py {shlex.join(train_argv)}

## 6. Evaluate on the test set

The `Benchmark` line covers the 33 category/tag pairs to compare with Liu et al. The per-tag table is saved to `My Drive/sidewalk/results/`.

In [ ]:
!python evaluate.py {CHECKPOINT} --data-dir {DATA_DIR} --output-dir {DRIVE_DIR}/results --num-workers {WORKERS}

## 7. Look at some predictions

In [ ]:
import random
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from dataset import CATEGORIES
from model import load_checkpoint
from predict import predict_tags

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, checkpoint = load_checkpoint(CHECKPOINT, device)

fig, axes = plt.subplots(2, 3, figsize=(15, 11))
for ax in axes.flat:
    category = random.choice(CATEGORIES)
    row = pd.read_csv(f'{DATA_DIR}/{category}/test.csv').sample(1).iloc[0]
    image = Image.open(f'{DATA_DIR}/{category}/test/{row.filename}')
    predicted = [tag for tag, _, passed in predict_tags(model, checkpoint, image, category, device) if passed]
    actual = [tag for tag in checkpoint['category_tags'][category] if row[tag] == 1]
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(f'{category}\npredicted: {", ".join(predicted) or "none"}\nactual: {", ".join(actual) or "none"}',
                 fontsize=9)
plt.tight_layout()
plt.show()